# Уникальные скороговорки датасета

Подход:
1. Транскрибируем **все** WAV-файлы через Whisper.
2. Исправляем ошибки транскрипции через LLM (Ollama).
3. Сохраняем все записи с текстом и контрольными буквами в `data/tongue_twisters.csv`.
4. Считаем статистику уникальных скороговорок.

In [ ]:
import sys
import re
from pathlib import Path
from collections import Counter
import numpy as np
import pandas as pd
import torch

exp_dir = Path().resolve()
sys.path.insert(0, str(exp_dir.parent))
from shared import config
from shared.data_utils import load_audio

CONTROL_LETTERS = list('лрстцчшщ')
print(f'DATA_DIR: {config.DATA_DIR}')

## 1. Загрузка Whisper

In [ ]:
from faster_whisper import WhisperModel

WHISPER_ID = "large"   # tiny / base / small / medium / large / large-v2 / large-v3
DEVICE     = "cuda"
GPU_INDEX  = 1         # номер GPU

model = WhisperModel(
    WHISPER_ID,
    device=DEVICE,
    device_index=GPU_INDEX,
    compute_type="float16",   # float16 для GPU, int8 для CPU
)
print(f"faster-whisper '{WHISPER_ID}' загружен на {DEVICE}:{GPU_INDEX}")

In [ ]:
# Диагностика: проверяем транскрипцию одного файла
import warnings
warnings.filterwarnings("ignore")

test_path = str(next(config.GOOD_DIR.glob('*.wav')))
segments, info = model.transcribe(test_path, language="ru", task="transcribe")
text = ' '.join(s.text for s in segments).strip()
print(f"Язык: {info.language} (вероятность {info.language_probability:.2f})")
print(f"Результат: {text}")

## 2. Вспомогательные функции

In [ ]:
import unicodedata

RUSSIAN_LOWER = frozenset(chr(c) for c in list(range(0x0430, 0x0450)) + [0x0451])

def fix_utf8_corrupted_on_windows(s):
    out = []
    i = 0
    while i < len(s):
        if i + 1 >= len(s):
            out.append(s[i]); i += 1; continue
        ord1, ord2 = ord(s[i]), ord(s[i+1])
        if ord2 < 0x80 or ord2 > 0xBF:
            if ord1 == 0x0430 and ord2 == 0x041B:
                byte2 = 0xBB
            else:
                out.append(s[i]); i += 1; continue
        else:
            byte2 = ord2
        first_byte = 0xD1 if ord1 == 0x0431 else (0xD0 if ord1 == 0x0430 else None)
        if first_byte is not None:
            try:
                out.append(bytes([first_byte, byte2]).decode('utf-8')); i += 2; continue
            except UnicodeDecodeError:
                pass
        out.append(s[i]); i += 1
    return ''.join(out)

def get_letter_combo(path):
    stem = Path(path).stem
    raw  = stem.split('__', 1)[1] if '__' in stem else ''
    decoded = fix_utf8_corrupted_on_windows(raw)
    return frozenset(c for c in decoded if c in RUSSIAN_LOWER)

def clean_text(text: str) -> str:
    """Удаляет спецсимволы: переносы строк, **, [n]."""
    text = str(text) if not isinstance(text, str) else text
    text = text.replace('\r\n', ' ').replace('\n', ' ').replace('\r', ' ')
    text = text.replace('**', '')
    text = re.sub(r'\[\d+\]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def transcribe(path: str) -> str:
    segments, _ = model.transcribe(
        str(path),
        language="ru",
        task="transcribe",
        no_speech_threshold=0.3,
        vad_filter=True,
    )
    return ' '.join(s.text for s in segments).strip()

## 3. Транскрипция всех файлов (Whisper)

In [ ]:
OUT_PATH  = config.DATA_DIR / 'tongue_twisters.csv'
col_order = ['filename', 'whisper_text', 'corrected_text'] + CONTROL_LETTERS

paths_good = sorted(config.GOOD_DIR.glob('*.wav'))
paths_bad  = sorted(config.BAD_DIR.glob('*.wav'))
all_paths  = [(str(p), config.CLASS_GOOD) for p in paths_good] + \
             [(str(p), config.CLASS_BAD)  for p in paths_bad]

print(f"Всего файлов: {len(all_paths)}")

# --- Загружаем существующий CSV или создаём пустой ---
if OUT_PATH.exists():
    df = pd.read_csv(OUT_PATH, encoding='utf-8')
    for col in ['whisper_text', 'corrected_text']:
        if col not in df.columns:
            df[col] = ''
        df[col] = df[col].fillna('').apply(clean_text)
    for letter in CONTROL_LETTERS:
        if letter not in df.columns:
            df[letter] = 0
    existing_filenames = set(df['filename'])
    print(f"Загружен существующий файл: {len(df)} записей")
else:
    df = pd.DataFrame(columns=col_order)
    existing_filenames = set()
    print("Файл не найден, создаём с нуля")

# --- Определяем, каким файлам нужен Whisper ---
need_whisper = []
new_rows = []
for path, label in all_paths:
    fname = Path(path).name
    letters = get_letter_combo(path)
    letter_vals = {letter: int(letter in letters) for letter in CONTROL_LETTERS}
    if fname in existing_filenames:
        wt = df.loc[df['filename'] == fname, 'whisper_text'].iloc[0]
        if wt == '':
            need_whisper.append((path, fname))
    else:
        need_whisper.append((path, fname))
        new_rows.append({'filename': fname, 'whisper_text': '', 'corrected_text': '', **letter_vals})

# Добавляем строки для новых файлов
if new_rows:
    df = pd.concat([df, pd.DataFrame(new_rows)], ignore_index=True)
    existing_filenames = set(df['filename'])

print(f"Файлов для Whisper: {len(need_whisper)}")

# --- Запускаем Whisper только на нужных файлах ---
errors_whisper = []
for i, (path, fname) in enumerate(need_whisper):
    try:
        whisper_text = clean_text(transcribe(path))
    except Exception as e:
        whisper_text = ''
        errors_whisper.append(f"{fname}: {e}")
    df.loc[df['filename'] == fname, 'whisper_text'] = whisper_text
    if (i + 1) % 100 == 0 or (i + 1) == len(need_whisper):
        print(f"  {i+1}/{len(need_whisper)}")

if errors_whisper:
    print(f"\nОШИБКИ ({len(errors_whisper)}):")
    for e in errors_whisper[:10]:
        print(f"  {e}")

print(f"\nПустых транскрипций: {(df['whisper_text'] == '').sum()}")
df.head(3)

## 4. Коррекция через Perplexity API

In [ ]:
import time
from openai import OpenAI

import os
API_KEY = os.environ["PERPLEXITY_API_KEY"]
pplx_client = OpenAI(
    api_key=API_KEY,
    base_url="https://api.perplexity.ai",
)

SYSTEM_PROMPT = (
    "Ты помогаешь восстанавливать русские скороговорки по искажённой записи. "
    "Отвечай только одной строкой — каноническим вариантом скороговорки без перевода и комментариев. "
    "Если не уверен или такой скороговорки не существует, напиши ровно: НЕ УВЕРЕН."
)

def correct_text(raw: str) -> str:
    raw = raw.strip()
    if not raw:
        return ''
    resp = pplx_client.chat.completions.create(
        model="sonar",
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user",   "content": raw},
        ],
        temperature=0.1,
        max_tokens=128,
    )
    return resp.choices[0].message.content.strip()

print("Perplexity API клиент готов.")
print(f"Файлов для коррекции: {len(df)}")

In [ ]:
def _len_ratio_bad(row) -> bool:
    """True если длины whisper и corrected сильно расходятся."""
    w = len(row['whisper_text'])
    c = len(row['corrected_text'])
    if w == 0 or c == 0:
        return False
    ratio = c / w
    return ratio < 0.4 or ratio > 3.0

corrected_col = df['corrected_text'].fillna('')
need_llm_mask = (
    (corrected_col == '') |
    corrected_col.str.contains('НЕ УВЕРЕН', na=False) |
    df.apply(_len_ratio_bad, axis=1)
)
need_llm_idx = df[need_llm_mask & (df['whisper_text'] != '')].index.tolist()

print(f"Строк для LLM коррекции: {len(need_llm_idx)}")
ratio_bad_count = df[need_llm_mask & (df['whisper_text'] != '') & (corrected_col != '') &
                     ~corrected_col.str.contains('НЕ УВЕРЕН', na=False)].shape[0]
print(f"  из них — несоответствие длин: {ratio_bad_count}")

errors_llm = 0
for count, i in enumerate(need_llm_idx):
    try:
        text = clean_text(correct_text(df.at[i, 'whisper_text']))
    except Exception as e:
        text = df.at[i, 'whisper_text']
        errors_llm += 1
        print(f"  Ошибка [{i}]: {e}")
    df.at[i, 'corrected_text'] = text
    time.sleep(0.3)
    if (count + 1) % 100 == 0 or (count + 1) == len(need_llm_idx):
        print(f"  {count+1}/{len(need_llm_idx)}  (ошибок: {errors_llm})")

print(f"\nОшибок API: {errors_llm}")
df[['whisper_text', 'corrected_text']].head(5)

## 5. Сохранение

In [ ]:
df_out = df[col_order].copy()

df_out.to_csv(OUT_PATH, index=False, encoding='utf-8')
print(f"Сохранено: {OUT_PATH}")
print(f"Записей: {len(df_out)}")
df_out.head(5)

## 6. Статистика

In [ ]:
import matplotlib.pyplot as plt

n_total       = len(df_out)
n_empty       = (df_out['whisper_text'] == '').sum()
n_unique_raw  = df_out[df_out['whisper_text'] != '']['whisper_text'].str.lower().str.strip().nunique()
n_unique_corr = df_out[df_out['corrected_text'] != '']['corrected_text'].str.lower().str.strip().nunique()

print("=" * 50)
print(f"Всего записей:            {n_total}")
print(f"Пустых (Whisper):         {n_empty}")
print(f"Уникальных (Whisper):     {n_unique_raw}")
print(f"Уникальных (после LLM):   {n_unique_corr}")
print("=" * 50)

# Топ-30 по частоте (по corrected_text)
top = (df_out[df_out['corrected_text'] != '']
       .groupby(df_out['corrected_text'].str.lower().str.strip())
       .agg(n=('filename', 'count'))
       .reset_index()
       .rename(columns={'corrected_text': 'text'})
       .sort_values('n', ascending=False)
       .head(30))

print(f"\nТоп-30 скороговорок:")
display(top.reset_index(drop=True))

counts_all = (df_out[df_out['corrected_text'] != '']
              .groupby(df_out['corrected_text'].str.lower().str.strip()).size()
              .sort_values(ascending=False))

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].bar(range(len(top)), top['n'], color='steelblue')
axes[0].set_xticks(range(len(top)))
axes[0].set_xticklabels([t[:25] for t in top['text']], rotation=90, fontsize=6)
axes[0].set_ylabel('Количество записей')
axes[0].set_title('Топ-30 скороговорок')

axes[1].hist(counts_all.values, bins=30, color='steelblue', edgecolor='white')
axes[1].set_xlabel('Записей на скороговорку')
axes[1].set_ylabel('Количество скороговорок')
axes[1].set_title(f'Распределение ({n_unique_corr} уникальных после LLM)')
axes[1].axvline(counts_all.mean(), color='red', linestyle='--',
                label=f'среднее {counts_all.mean():.1f}')
axes[1].legend()

plt.tight_layout()
plt.savefig(exp_dir / 'tongue_twisters_stats.png', dpi=150, bbox_inches='tight')
plt.show()